<a href="https://colab.research.google.com/github/leoncana/CreativeHub-DataEngineeringModule1/blob/main/notebooks/session06_pipeline_medallion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6 — Building a data pipeline: bronze, silver, gold

So far we wrote analysis code in cells and ran it by hand. Real data engineering is about **pipelines**:
code that runs every day, on its own, and produces tables people can **trust**.

**Today you will build Wavely's first real pipeline**
1. **Bronze** — land the raw files exactly as they are (plus where/when they came from)
2. **Silver** — clean, typed, deduplicated data; bad rows go to **quarantine**
3. **Gold** — business tables that answer questions
4. **Quality checks** — the pipeline tests itself
5. Wrap everything in functions and prove the pipeline is **idempotent** (safe to re-run)

## Homework review (15 min)
Open `homework/hw05_spark_analysis.ipynb`.

---
## Part 1 — Concepts

**ETL vs ELT**
- **ETL** (Extract → Transform → Load): clean the data *before* storing it. Classic, but if your cleaning had a bug, the raw data is gone.
- **ELT** (Extract → Load → Transform): store the raw data first, transform later inside the data platform. This is the modern default.

**The medallion architecture** (used at Databricks and many other companies):

```
 raw CSV files ──► BRONZE ──────────────► SILVER ───────────────────► GOLD
                   as-is, all text,        typed, deduplicated,         aggregated business tables:
                   + _source_file          validated; bad rows ──►      daily listening, top tracks,
                   + _ingested_at          QUARANTINE                   user activity
```

**Two promises a good pipeline makes**
- **Lineage:** for every row we can say where it came from (which file, when it was loaded).
- **Idempotency:** running the pipeline twice gives exactly the same result as running it once. No double counting — artists get paid per play!

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


In [2]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 4.0.5 is ready


In [3]:
import shutil

LAKE = DATA.parent / "lake"  # our mini "data lake": bronze/, silver/, gold/
shutil.rmtree(LAKE, ignore_errors=True)  # start fresh for the lesson
print("Lake folder:", LAKE)

Lake folder: /content/course/lake


---
## Part 2 — Bronze: land the raw data

Rules for bronze:
- Read **every column as a string**. If a value can't be parsed, we must not lose the row.
- **Change nothing** in the data.
- Add lineage columns: `_source_file` (the file name) and `_ingested_at` (the load time).

Spark gives you the file each row came from in a hidden column: `F.col("_metadata.file_name")`.

### Exercise 1 (Core) — `extract_to_bronze`
Complete the function: read all `raw/plays_*.csv` files with `RAW_SCHEMA`, add the two lineage columns,
and write the result as Parquet to `LAKE / "bronze" / "plays"` with mode `overwrite`. Return the number of rows written.

In [4]:
RAW_SCHEMA = (
    "event_id STRING, event_ts STRING, user_id STRING, track_id STRING, ms_played STRING, "
    "device STRING, country STRING, skipped STRING, shuffle STRING"
)


def extract_to_bronze():
    # YOUR CODE HERE
    raw = (
        spark.read.csv(str(DATA/'raw'/'plays_*.csv'), header=True, schema=RAW_SCHEMA)
        .withColumn('_source_file', F.col('_metadata.file_name'))
        .withColumn('_ingested_at', F.current_timestamp())
    )

    raw.write.mode('overwrite').parquet(str(LAKE / "bronze" / "plays"))

    return spark.read.parquet(str(LAKE / "bronze" / "plays")).count()


print(extract_to_bronze(), "rows in bronze")  # expected: 60823
bronze = spark.read.parquet(str(LAKE / "bronze" / "plays"))
bronze.groupBy("_source_file").count().orderBy("_source_file").show(3, truncate=False)

60823 rows in bronze
+--------------------+-----+
|_source_file        |count|
+--------------------+-----+
|plays_2026-09-01.csv|4334 |
|plays_2026-09-02.csv|3902 |
|plays_2026-09-03.csv|4338 |
+--------------------+-----+
only showing top 3 rows


---
## Part 3 — Silver: clean, typed, trustworthy

Silver rules for Wavely plays:
1. Cast to proper types: `event_ts` → timestamp, `ms_played` → int, `skipped` / `shuffle` → boolean
2. One row per `event_id` (the app sometimes sends an event twice)
3. A row is **valid** if `event_ts` and `user_id` are not null and `ms_played >= 0`
4. Valid rows → `silver/plays` (with extra columns `play_date` and `minutes_played`), partitioned by `play_date`
5. Invalid rows → `silver/plays_quarantine`: we never silently delete data; someone can investigate later

### Exercise 2 (Core) — `build_silver`
The typing step is done for you. Complete the deduplication, the valid/invalid split and the writes.

In [6]:
def build_silver():
    bronze = spark.read.parquet(str(LAKE / "bronze" / "plays"))
    typed = (
        bronze
        .withColumn("event_ts", F.to_timestamp("event_ts", "yyyy-MM-dd HH:mm:ss"))
        .withColumn("ms_played", F.col("ms_played").cast("int"))
        .withColumn("skipped", F.col("skipped").cast("boolean"))
        .withColumn("shuffle", F.col("shuffle").cast("boolean"))
    )
    is_valid = F.col("event_ts").isNotNull() & F.col("user_id").isNotNull() & (F.col("ms_played") >= 0)

    deduped = typed.dropDuplicates(['event_id'])

    valid = (
        deduped.filter(is_valid)
        .withColumn('play_date', F.to_date('event_ts'))
        .withColumn('minutes_played', F.round(F.col('ms_played')/ 60000, 2))
    )
    valid.write.partitionBy('play_date').mode('overwrite').parquet(str(LAKE / "silver" / "plays"))

    invalid = deduped.filter(~is_valid)
    invalid.write.mode('overwrite').parquet(str(LAKE / "silver" / "plays_quarantine"))

build_silver()
silver = spark.read.parquet(str(LAKE / "silver" / "plays"))
quarantine = spark.read.parquet(str(LAKE / "silver" / "plays_quarantine"))
print(silver.count(), "rows in silver,", quarantine.count(), "in quarantine")  # expected: 59801 rows in silver, 130 in quarantine
silver.printSchema()

59801 rows in silver, 130 in quarantine
root
 |-- event_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- ms_played: integer (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- skipped: boolean (nullable = true)
 |-- shuffle: boolean (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- play_date: date (nullable = true)



> **Under the hood:** look at the folder `lake/silver/plays/`. There is one sub-folder per day (`play_date=2026-09-01/`, ...).
> Spark stores the partition value in the folder name, not inside the files. Queries that filter on `play_date` only open the folders they need.

In [7]:
sorted(p.name for p in (LAKE / "silver" / "plays").iterdir())[:5]

['._SUCCESS.crc',
 '_SUCCESS',
 'play_date=2026-09-01',
 'play_date=2026-09-02',
 'play_date=2026-09-03']

In [8]:
silver.show()

+------------+-------------------+-------+--------+---------+-------------+-------+-------+-------+--------------------+--------------------+--------------+----------+
|    event_id|           event_ts|user_id|track_id|ms_played|       device|country|skipped|shuffle|        _source_file|        _ingested_at|minutes_played| play_date|
+------------+-------------------+-------+--------+---------+-------------+-------+-------+-------+--------------------+--------------------+--------------+----------+
|0037e5abd0e2|2026-09-05 12:02:00|  U0341|   T0002|    46722|       mobile|     DE|  false|  false|plays_2026-09-05.csv|2026-11-24 17:53:...|          0.78|2026-09-05|
|00423e1206c4|2026-09-05 11:18:12|  U0146|   T0018|    23484|smart_speaker|     CZ|   true|  false|plays_2026-09-05.csv|2026-11-24 17:53:...|          0.39|2026-09-05|
|00456963c79c|2026-09-05 22:14:11|  U0683|   T0011|   182000|       mobile|     CZ|  false|  false|plays_2026-09-05.csv|2026-11-24 17:53:...|          3.03|2026

---
## Part 4 — Gold: tables for the business

Gold tables answer questions directly. They are small, so dashboards load them instantly.
Here is the first one, built together:

In [9]:
def read_reference(name):
    return spark.read.csv(str(DATA / "reference" / f"{name}.csv"), header=True, inferSchema=True)


def build_gold_daily_country(silver):
    return (
        silver
        .groupBy("play_date", "country")
        .agg(
            F.count("*").alias("plays"),
            F.countDistinct("user_id").alias("listeners"),
            F.round(F.sum("ms_played") / 3_600_000, 2).alias("hours"),
        )
    )


build_gold_daily_country(silver).orderBy("play_date", "country").show(6)

+----------+-------+-----+---------+-----+
| play_date|country|plays|listeners|hours|
+----------+-------+-----+---------+-----+
|2026-09-01|     AT| 1152|      258|45.56|
|2026-09-01|     CZ|  597|      130|24.81|
|2026-09-01|     DE|  811|      144|33.58|
|2026-09-01|     HU|  486|      110|18.85|
|2026-09-01|     PL|  496|       98|18.55|
|2026-09-01|     SK|  714|      141|28.76|
+----------+-------+-----+---------+-----+
only showing top 6 rows


### Exercise 3 (Core) — Two more gold tables
1. `build_gold_top_tracks(silver)` — one row per track with `track_id`, `title`, `artist_name`, `genre`,
   `streams` (plays that were **not skipped** — that's what artists get paid for), `listeners` (distinct users) and `hours`.
2. `build_gold_user_activity(silver)` — one row per user with `user_id`, `plan`, `plays`, `hours`, `skip_rate` (percent, 1 decimal)
   and `active_days` (distinct `play_date`s).

In [21]:
def build_gold_top_tracks(silver):
    tracks = read_reference('tracks').select('track_id', 'title', 'artist_id', 'genre')
    artists = read_reference('artists').select('artist_id', 'artist_name')

    gold_top_tracks = (
        silver
        .join(tracks, on = 'track_id')
        .join(artists, on = 'artist_id')
        .filter(~F.col('skipped'))
        .groupBy('track_id','title', 'artist_name', 'genre')
        .agg(
            F.count("*").alias('streams'),
            F.countDistinct('user_id').alias('listeners'),
            F.round(F.sum('ms_played')/3600000, 2).alias('hours'),
        )
        .select('track_id', 'title', 'artist_name', 'genre', 'streams', 'listeners', 'hours')
    )
    return gold_top_tracks

def build_gold_user_activity(silver):
    # YOUR CODE HERE
    users = read_reference('users').select('user_id', 'plan')
    user_activity = (
        silver.join(users, on='user_id', how='left')
        .groupBy('user_id', 'plan')
        .agg(
            F.count("*").alias('plays'),
            F.round(F.sum('ms_played')/3600000, 2).alias('hours'),
            F.round(F.avg(F.col('skipped').cast('int')), 1).alias('skip_rate'),
            F.countDistinct('play_date').alias('active_days')

        )
        .select('user_id', 'plan', 'plays', 'hours', 'skip_rate', 'active_days')
    )

    return user_activity


build_gold_top_tracks(silver).orderBy(F.desc("streams")).show(5)
build_gold_user_activity(silver).orderBy(F.desc("hours")).show(5)

+--------+--------------+------------+-----+-------+---------+-----+
|track_id|         title| artist_name|genre|streams|listeners|hours|
+--------+--------------+------------+-----+-------+---------+-----+
|   T0061|  Faded Lights|   Bad Bunny|latin|    783|      260|42.93|
|   T0006|Northern Ocean|Taylor Swift|  pop|    736|      352|42.61|
|   T0020|Crystal Lights|  The Weeknd|  pop|    679|      313|29.31|
|   T0013|  Little Stars|    Dua Lipa|  pop|    674|      329|32.73|
|   T0140|     Wild City|    Coldplay| rock|    667|      236|52.19|
+--------+--------------+------------+-----+-------+---------+-----+
only showing top 5 rows
+-------+-------+-----+-----+---------+-----------+
|user_id|   plan|plays|hours|skip_rate|active_days|
+-------+-------+-----+-----+---------+-----------+
|  U0156|   free|  520|29.09|      0.2|         14|
|  U0622|premium|  545|26.25|      0.2|         14|
|  U0483|premium|  601|25.91|      0.3|         14|
|  U0698|premium|  481|19.65|      0.3|    

Now one function that builds and writes all gold tables:

In [22]:
def build_gold():
    silver = spark.read.parquet(str(LAKE / "silver" / "plays"))
    tables = {
        "daily_country": build_gold_daily_country(silver),
        "top_tracks": build_gold_top_tracks(silver),
        "user_activity": build_gold_user_activity(silver),
    }
    for name, df in tables.items():
        df.write.mode("overwrite").parquet(str(LAKE / "gold" / name))


build_gold()
for name in ["daily_country", "top_tracks", "user_activity"]:
    print(name, spark.read.parquet(str(LAKE / "gold" / name)).count(), "rows")

daily_country 84 rows
top_tracks 186 rows
user_activity 800 rows


---
## Part 5 — Quality checks: the pipeline tests itself

A check is a question whose answer should be **"zero bad rows"**. If a check fails, the pipeline stops
**before** bad numbers reach a dashboard (or an artist's payment).

In [23]:
def check(name, bad_rows):
    """A check passes when the DataFrame of bad rows is empty."""
    n_bad = bad_rows.count()
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}  ({n_bad} bad rows)")
    return n_bad == 0


silver = spark.read.parquet(str(LAKE / "silver" / "plays"))
check("event_id is unique in silver", silver.groupBy("event_id").count().filter(F.col("count") > 1))
check("no negative ms_played", silver.filter(F.col("ms_played") < 0))

PASS  event_id is unique in silver  (0 bad rows)
PASS  no negative ms_played  (0 bad rows)


True

### Exercise 4 (Core) — `run_quality_checks`
Add these checks to the function and make it **raise an error** if any check fails
(`raise ValueError("...")` stops the program with a message):

1. every `track_id` in silver exists in `tracks.csv` (hint: `left_anti` join)
2. no play is longer than its track (`ms_played > duration_ms` after joining tracks)
3. **reconciliation:** total hours in `gold/daily_country` equals total hours in silver. Gold values are rounded, so allow a difference of up to 1 hour.
   Hint: build a small DataFrame of bad rows with `spark.createDataFrame([(a, b)], "gold DOUBLE, silver DOUBLE").filter("abs(gold - silver) > 1")`

In [ ]:
def run_quality_checks():
    silver = spark.read.parquet(str(LAKE / "silver" / "plays"))
    tracks = read_reference("tracks")
    results = [
        check("event_id is unique in silver", silver.groupBy("event_id").count().filter(F.col("count") > 1)),
        check("no negative ms_played", silver.filter(F.col("ms_played") < 0)),
    ]
    # YOUR CODE HERE

    results.append()

    gold_hours = spark.read.parquet()
    ...


run_quality_checks()

**See a check fail:** what if the deduplication step had a bug? Let's simulate it by appending a few rows twice:

In [ ]:
broken = silver.unionByName(silver.limit(3))  # 3 duplicate events
check("event_id is unique (broken data)", broken.groupBy("event_id").count().filter(F.col("count") > 1))

---
## Part 6 — The whole pipeline, twice

In [ ]:
def run_pipeline():
    n = extract_to_bronze()
    build_silver()
    build_gold()
    run_quality_checks()
    total = spark.read.parquet(str(LAKE / "gold" / "daily_country")).agg(F.sum("plays")).first()[0]
    print(f"bronze rows: {n}, plays in gold: {total}")
    return total


first = run_pipeline()
second = run_pipeline()
print("Idempotent:", first == second)

Because every step **overwrites** its output and reads only from the previous layer, running it again gives the same result.
If step 2 had used `mode("append")`, the second run would have doubled every number. Try it if you like!

> **Under the hood:** in a real platform, `run_pipeline()` is started by a **scheduler** every night (session 9),
> and the layers live in cloud storage (S3, Azure Data Lake, Google Cloud Storage) instead of a local folder.

---
## Part 7 — Show it

Gold tables are small, so it is fine to bring them to the driver with `toPandas()` and plot them.
Never do this with a big silver table!

In [ ]:
daily = (
    spark.read.parquet(str(LAKE / "gold" / "daily_country"))
    .groupBy("play_date")
    .pivot("country")
    .sum("hours")
    .orderBy("play_date")
    .toPandas()
    .set_index("play_date")
)
ax = daily.plot(figsize=(10, 4), marker="o", title="Listening hours per day and country")
ax.set_ylabel("hours");

**Discuss:** which days are the peaks? Why?

## Wrap-up

| Concept | In our pipeline |
|---|---|
| bronze | raw strings + `_source_file` + `_ingested_at` |
| silver | typed, deduplicated, valid rows; invalid → quarantine; partitioned by `play_date` |
| gold | `daily_country`, `top_tracks`, `user_activity` |
| quality checks | `check(name, bad_rows)`, reconciliation, stop on failure |
| idempotency | `mode("overwrite")`, run twice → same numbers |

**Homework:** `homework/hw06_pipeline.ipynb`